# 12-filter — Bộ lọc cũ/mới trước lượt S

**Câu hỏi**: bộ lọc của bước 12 giữ và loại những ảnh pool nào, nó quyết định bằng số hạng nào của log-mật độ (b, q, ℓ như `08`), khác re-labeling của paper (Algorithm 1) ở đâu, và sau bộ lọc, nhãn mà lượt S học sạch hơn nhãn của lượt F bao nhiêu (cột 12 của sổ mất điểm)? (D12, D25, Q08.02, Q12.02; QD24)

### Nguồn gốc

Mẫu cell 0 (`lab/README.vi.md` §11): in commit và trạng thái dirty, rồi đặt `XLA_FLAGS` ở chế độ GPU tất định trước mọi lần import jax.

In [1]:
# cell 0: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])

commit 4f8166d | code dirty False | XLA_FLAGS --xla_gpu_deterministic_ops=true


- **Kết quả**: commit `4f8166d`, code dirty False, XLA_FLAGS `--xla_gpu_deterministic_ops=true`.

## 1. Kiểm

### 1.1 Chạy lại bước 12 ở r0 s4; số ảnh giữ ở 15 ảnh chụp

Đầu ra của bước 12 là `keep`. Với mỗi ảnh trong 748 ảnh pool, `keep` là `True` khi bộ phân lớp của lượt F (đủ 10 khe, tham số `params_f`) đặt ảnh vào một khe mới (5–9). Ảnh được giữ đi vào lượt S với id cụm của bước 09 (D12).

Chạy lại bước 12 của `gc10_v2_ft_r0_s4` từ ảnh chụp 11 bằng `replay.rerun`, rồi so `keep` với ảnh chụp 12. Đây là khoá rời rạc nên phải trùng tuyệt đối. In thêm dòng log `Number of Novel Samples` mà bước 12 in ra, giống `mngmm.train`. Sau đó đọc `keep` trong `snap/12.pkl` của 15 ảnh chụp, đếm số ảnh giữ và so với khoá `S1_novel_kept` trong `metrics.json` của chính run đó.

Số giữ đã có trong `metrics.json`, và r0 s4 đã được đo ở `04_s1_discovery` (tag `nb-v1`, cell 7: 689/748). Vì vậy cell này là phép kiểm nhất quán, không phải dự đoán mù.

Dự đoán [SUY LUẬN]:
- `rerun` OK 1/1 khoá, 0 ảnh khác. Lý do: bước 12 chỉ lấy argmax của log-mật độ với `params_f` và không rút RNG; `params_f` đã tái lập từng bit ở `11` §1.1 dù driver GPU đổi;
- dòng log là 689 / 748;
- số giữ trùng `S1_novel_kept` ở 15/15.

In [2]:
# 1.1 rerun step 12 (pass-F classifier on the pool, keep the images put in a novel slot) and compare with snap/12.pkl;
#     kept counts of the 15 snapshots against S1_novel_kept in their metrics.json
import io, contextlib, json, time
import numpy as np

RUN = "gc10_v2_ft_r0_s4"
buf, t0 = io.StringIO(), time.time()
with contextlib.redirect_stdout(buf):
    out, ok = replay.rerun(RUN, "12")
log = buf.getvalue()
print("\n".join(l for l in log.splitlines() if l.startswith(("rerun", "  OK", "  DIFF", "Number of Novel Samples"))))
print(f"time {time.time() - t0:.0f} s | keep: {out['keep'].dtype}, shape {out['keep'].shape}, kept {int(out['keep'].sum())}")

RUNS = [f"gc10_v2_ft_r{r}_s{s}" for r in range(3) for s in range(5)]
kept = {}
for run in RUNS:
    k = np.asarray(replay.snapshots(run)["12"]["out"]["keep"])
    m = json.loads((replay._run_dir(run) / "metrics.json").read_text())
    kept[run.replace("gc10_v2_ft_", "")] = (int(k.sum()), len(k), m["S1_novel_kept"])
print("\nkept / pool (S1_novel_kept) per snapshot:")
for r in range(3):
    print("  " + " | ".join(f"{t} {a}/{n} ({b})" for t, (a, n, b) in kept.items() if t.startswith(f"r{r}")))
print("kept = S1_novel_kept in", sum(a == b for a, _, b in kept.values()), "/ 15")
print("kept - 698:", {t: a - 698 for t, (a, _, _) in kept.items()})

/home/rito/miniforge3/envs/vbcgcd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rerun step 12 of gc10_v2_ft_r0_s4 | device gpu | GPU now NVIDIA GeForce RTX 4060 Laptop GPU, 621.01 | GPU of the run NVIDIA GeForce RTX 4060 Laptop GPU, 617.42 | driver differs: float outputs may move in the last bits
Number of Novel Samples: 689 / 748
  OK   keep         array bool             max|diff| 0
rerun step 12: OK (1/1 keys)
time 10 s | keep: bool, shape (748,), kept 689

kept / pool (S1_novel_kept) per snapshot:
  r0_s0 701/748 (701) | r0_s1 696/748 (696) | r0_s2 693/748 (693) | r0_s3 695/748 (695) | r0_s4 689/748 (689)
  r1_s0 693/748 (693) | r1_s1 691/748 (691) | r1_s2 688/748 (688) | r1_s3 696/748 (696) | r1_s4 687/748 (687)
  r2_s0 702/748 (702) | r2_s1 700/748 (700) | r2_s2 699/748 (699) | r2_s3 688/748 (688) | r2_s4 695/748 (695)
kept = S1_novel_kept in 15 / 15
kept - 698: {'r0_s0': 3, 'r0_s1': -2, 'r0_s2': -5, 'r0_s3': -3, 'r0_s4': -9, 'r1_s0': -5, 'r1_s1': -7, 'r1_s2': -10, 'r1_s3': -2, 'r1_s4': -11, 'r2_s0': 4, 'r2_s1': 2, 'r2_s2': 1, 'r2_s3': -10, 'r2_s4': -3}


- **Kết quả**:
  - `rerun` bước 12 ở r0 s4: OK 1/1 khoá. `keep` là mảng bool 748 phần tử, không ảnh nào khác ảnh chụp, dù driver GPU đã đổi (617.42 → 621.01). Thời gian 10 s.
  - Dòng log: `Number of Novel Samples: 689 / 748`.
  - Số giữ trùng `S1_novel_kept` ở 15/15: r0 701, 696, 693, 695, 689; r1 693, 691, 688, 696, 687; r2 702, 700, 699, 688, 695.
  - Hiệu (số giữ − 698), với 698 là số ảnh lớp mới trong pool:
    - dương ở 4 ảnh chụp: r0 s0 (+3), r2 s0 (+4), r2 s1 (+2), r2 s2 (+1);
    - âm ở 11 ảnh chụp còn lại, từ −2 tới −11.
- **Ý nghĩa**:
  - Đúng như dự đoán ở cả ba ý. Bước 12 tái lập được, và ảnh chụp 12 khớp số mà pipeline ghi.
  - Số giữ = 698 − (số ảnh mới bị loại) + (số ảnh cũ lọt). Ở bốn ảnh chụp có hiệu dương, chắc chắn có ảnh cũ lọt, ít nhất 1–4 ảnh.
  - Riêng hiệu này không tách được hai phần; §3.1 tách bằng nhãn thật.

## 2. Code

### 2.1 Bộ lọc trong code và trong paper; re-labeling của paper sẽ đổi bao nhiêu nhãn

In code của ba hàm bằng `inspect.getsource`: `s12_filter`, `s13_gauss_s` (để thấy lượt S học nhãn nào) và `MNGMMClassifier.train`. Đặt chúng cạnh Algorithm 1 của paper (Phụ lục A):
- dòng 30–33 (lượt F): SVI trên nhãn cụm ŷ, ra μ′, Σ′ của các lớp mới;
- dòng 34–36 (re-labeling): ȳ_i = classifier(h(x_i); μ ∪ μ′, Σ ∪ Σ′) cho mọi ảnh của phiên;
- dòng 37–40 (lượt S): SVI lại trên (x_i, ȳ_i), ghi vào μ[ȳ_i], Σ[ȳ_i]. Theo cách đọc ở D12, việc này áp cả khi ȳ_i là lớp cũ.

Sau đó, không dùng nhãn thật. Với mỗi ảnh chụp, lấy khe mà bộ phân lớp của lượt F chọn cho từng ảnh pool (`_predict` với `params_f`, như `s12_filter` làm). Chia 748 ảnh thành ba nhóm:
- bị loại: ảnh vào một khe cũ (0–4). Paper sẽ gán ảnh này cho lớp cũ đó;
- giữ, đúng khe của cụm mình (id cụm + 5): code và paper cho cùng nhãn;
- giữ, nhưng ở một khe mới khác: code giữ id cụm, paper đổi sang khe mới đó.

In bảng cụm × khe ở r0 s4. Hai phép kiểm:
- khe ≥ 5 phải trùng `keep` của ảnh chụp 12, ở 15/15;
- nhóm "đúng khe" phải gần `train_acc` mà lượt F in ra ở bước 900 (log nằm trong ảnh chụp 11). Số ở bước 900 tính trên tham số sau 901 lần cập nhật, nên lệch vài ảnh là bình thường.

Dự đoán [SUY LUẬN]:
- code chỉ lọc: lượt S học id cụm của các ảnh được giữ, và không ảnh nào đi vào khe cũ (D12);
- `keep` trùng ở 15/15; nhóm "đúng khe" lệch log bước 900 không quá 10 ảnh;
- nhóm "giữ, ở khe mới khác" chiếm khoảng 5–20% pool (40–150 ảnh), cao nhất ở r1. Lý do:
  - bộ phân lớp quyết định bằng mẫu dấu của hiệp phương sai (số hạng q, `08` §3);
  - k-means thì chia theo vị trí trên 2–3 hướng đầu của pool (`09` §4.2);
  - trên chính pool mà bộ phân lớp đã học, nó lệch k-means ít hơn trên test (`11` §4.2);
- ở r0 s4, ảnh bị loại đến nhiều nhất từ cụm 1 (waist) và cụm 0 (inclusion), vì 35/50 ảnh cũ nằm ở hai cụm này (`09d` §2.1). Cụm 1 đổ chủ yếu vào khe punching và welding, vì 10 ảnh punching và 9 ảnh welding của pool nằm ở cụm này.

In [3]:
# 2.1 the filter in the code (s12_filter, s13_gauss_s, MNGMMClassifier.train); then, without true labels, the slot the
#     pass-F classifier picks for each pool image: dropped (old slot), kept in its own cluster's slot, or kept in another
#     novel slot (a label that the paper's re-labeling would change)
import inspect, re
import jax.numpy as jnp
import pandas as pd
from classifier.mngmm import MNGMMClassifier
for f in (replay.s12_filter, replay.s13_gauss_s, MNGMMClassifier.train):
    print(f"# ---- {f.__qualname__}\n{inspect.getsource(f)}")

def pass_f_slots(run):
    """Slot (0-9) that the pass-F classifier gives each pool image (mngmm._predict with params_f, as s12_filter does),
    with the cluster ids of step 09 and the offset (first novel slot)."""
    st = replay.load(run, "11", check=False)
    replay.predict_default(st["clf"])
    slot = np.asarray(st["clf"]._predict(jnp.asarray(st["z_pool"]), replay._jnp(st["params_f"]))[0])
    return slot, np.asarray(st["ids"]).astype(int), int(st["offset"])

rows21 = []
for run in RUNS:
    slot, ids, off = pass_f_slots(run)
    b = replay.snapshots(run)
    own = slot == ids + off
    log_acc = re.findall(r"train_acc = (\d+)/\d+", b["11"]["log"])          # pass-F log lines, every 100 steps
    rows21.append(dict(snap=run.replace("gc10_v2_ft_", ""), keep_ok=bool(np.array_equal(slot >= off, b["12"]["out"]["keep"])),
                       dropped=int((slot < off).sum()), kept_own=int(own.sum()), kept_other=int(((slot >= off) & ~own).sum()),
                       log_own_step900=int(log_acc[-1]) if log_acc else -1))
    if run == RUN:
        xt21 = pd.crosstab(pd.Series(ids, name="cluster"), pd.Series(slot, name="slot")).reindex(columns=range(10), fill_value=0)
T21 = pd.DataFrame(rows21).set_index("snap")
print("r0 s4, pool images: cluster of step 09 (rows) x slot of the pass-F classifier (columns; 0-4 old = dropped, own slot = cluster + 5):")
print(xt21.to_string())
print("\n15 snapshots (748 pool images each):")
print(T21.to_string())
print("\nkept_other / 748 by FT seed (min, max):",
      {ft: (round(float(g.min()) / 748, 3), round(float(g.max()) / 748, 3)) for ft, g in T21["kept_other"].groupby(T21.index.str[:2])})

# ---- s12_filter
def s12_filter(st):
    """12 Old/new filter: keep the pool images that the pass-F classifier puts in a novel slot (mngmm.train).
    In: z_pool, params_f, predict, offset. Out: keep (bool per pool image)."""
    import jax.numpy as jnp
    clf, off = _clf(st), st["offset"]
    _component(st["predict"])(clf)
    clf.label_offset, clf.n_seen = off, off + st["args"].increment
    pred, _ = clf._predict(jnp.array(st["z_pool"]), _jnp(st["params_f"]))
    novel = pred >= off
    print(f"Number of Novel Samples: {novel.sum()} / {len(st['z_pool'])}")
    return dict(keep=np.asarray(novel))

# ---- s13_gauss_s
def s13_gauss_s(st):
    """13 Pass S: SVI on the kept pool images, started again from the S0 Gaussians (mngmm.train).
    In: z_pool, ids, keep, params0, fit, stop; z_test1, true_test1_y, lut (log lines only). Out: params1."""
    clf, a, off = _clf(st), st["args"], st["offset"]
    _component(st["stop"])(clf, a)
    clf._set_label_offset(off)
    clf.global_params, cl

- **Kết quả**:
  - Code:
    - `s12_filter` giữ ảnh có khe dự đoán ≥ offset;
    - `s13_gauss_s` học trên `z_pool[keep]` với nhãn `(ids + offset)[keep]`, tức id cụm của bước 09, không phải khe dự đoán;
    - `train()` làm đúng như vậy (`features[novel_idx]`, `labels[novel_idx]`). Khe cũ không nhận ảnh nào ở lượt S.
  - Khe ≥ 5 trùng `keep` ở 15/15.
  - 748 ảnh pool chia ba nhóm:
    - bị loại 46–61;
    - giữ, đúng khe của cụm mình 670–693 (89.6–92.6%);
    - giữ, ở khe mới khác chỉ 7–21 ảnh (0.9–2.8% pool). Theo FT seed: r0 1.9–2.8%, r1 1.5–2.3%, r2 0.9–1.7%.
  - Log `train_acc` ở bước 900 thấp hơn nhóm "đúng khe" ở bước 1000 từ 3 đến 9 ảnh.
  - r0 s4, bảng cụm × khe:
    - cụm 1 (waist): mất 29 ảnh vào khe cũ (punching 12, welding 8, crescent 5, water 2, oil 2);
    - cụm 0 (inclusion): mất 17 ảnh (crescent 6, water 4, oil 6, welding 1), thêm 10 ảnh sang khe mới khác (7 sang cụm 2);
    - cụm 2 (silk lớn): mất 12 ảnh (water 6, oil 6);
    - cụm 3: mất 1 ảnh; cụm 4: giữ đủ 103/103.
- **Ý nghĩa**:
  - Đúng ở code (D12), ở phép kiểm `keep`, ở độ lệch log (≤ 10 ảnh), và ở chỗ cụm 1 và cụm 0 mất nhiều nhất (46/59 ảnh).
  - Sai ở ý chính. Dự đoán là 5–20% ảnh ở khe mới khác; thực tế chỉ 0.9–2.8%. Dự đoán "cao nhất ở r1" cũng sai: r0 cao nhất. Trên chính pool mà nó học, bộ phân lớp của lượt F lặp lại phân hoạch k-means ở khoảng 90% ảnh.
  - Hệ quả cho Q12.03: nếu dùng re-labeling như paper, nhãn giữa các khe mới chỉ đổi 7–21 ảnh. Khác biệt lớn giữa paper và code là phần D12 (b): 46–61 ảnh bị loại sẽ được gán cho lớp cũ thay vì bị bỏ.
  - [SUY LUẬN] Bộ phân lớp khớp k-means trên pool (khoảng 90%) nhưng lệch nhiều trên test (`11` §4.2). Vậy chỗ mất 10 → 11 nhiều khả năng nằm ở khâu chuyển từ pool sang test (cụm bám cuộn, Q10.03), không ở việc bộ phân lớp không học nổi cụm.
  - [SUY LUẬN, ghép `nb-v1` cell 7 với `09d` §2.1] Cụm 1 có 19 ảnh cũ mà mất 29 ảnh, nên ít nhất 10 ảnh mới bị loại nằm ở cụm 1; §3.1 kiểm.

## 3. Bộ lọc so với nhãn thật

### 3.1 Ảnh cũ bị loại hay lọt, ảnh mới bị loại theo lớp

Nhãn thật của pool (`true_pool_y`) chỉ dùng để chấm. Với 15 ảnh chụp, đếm bốn thứ:
- số ảnh cũ bị loại và lọt (trên 50 ảnh);
- số ảnh mới bị loại theo từng lớp mới;
- lớp cũ nào lọt;
- khe cũ nào nhận ảnh mới bị loại.

In thêm hai bảng:
- r0 s4 theo từng cụm: số ảnh cũ, số bị loại;
- cụm do crease sở hữu ở r0 s2 (QD24; cụm này có 34/93 ảnh cũ, `09d` §7), trước và sau bộ lọc.

r0 s4 đã được đo ở `04_s1_discovery` (`nb-v1`, cell 7): lọt 2/50 ảnh cũ (water), loại 11 ảnh mới (inclusion 8, rolled 2, waist 1).

Dự đoán [SUY LUẬN]:
- ảnh cũ bị loại 44–50/50 ở mọi ảnh chụp; số lọt không nhỏ hơn cận của 1.1 (r0 s0 ≥ 3, r2 s0 ≥ 4);
- ảnh mới bị loại 3–15 mỗi ảnh chụp, inclusion nhiều nhất. Tỷ lệ bị loại theo lớp: inclusion 3–8%, rolled 0–12%, crease 0–6%, waist khoảng 1%, silk gần 0;
- ảnh mới bị loại chủ yếu vào khe oil và punching, như trên test (`11` §4.2);
- r0 s4: cụm 1 loại đủ 19 ảnh cũ cộng khoảng 10 ảnh mới (2.1);
- r0 s2: bộ lọc loại ít nhất 28/34 ảnh cũ của cụm crease và giữ gần hết crease.

In [4]:
# 3.1 the filter against the true pool labels (scoring only), 15 snapshots: old images dropped or leaked, novel images
#     dropped per class and the old slot that takes them; r0 s4 per cluster; the crease-owned cluster of r0 s2 (QD24)
import json
from collections import Counter
split = json.load(open(ROOT / "splits" / "gc10_split.json"))
names = [c.split("_")[0] for c in split["classes"]]

rows31, leak, to_old, POOL = [], Counter(), Counter(), {}
for run in RUNS:
    b = replay.snapshots(run)
    y = np.asarray(b["04"]["out"]["true_pool_y"]).astype(int)
    ids = np.asarray(b["09"]["out"]["ids"]).astype(int)
    keep = np.asarray(b["12"]["out"]["keep"]).astype(bool)
    slot, _, off = pass_f_slots(run)
    POOL[run] = dict(y=y, ids=ids, keep=keep, slot=slot)
    old = y < off
    r = dict(snap=run.replace("gc10_v2_ft_", ""), old_dropped=int((old & ~keep).sum()), old_kept=int((old & keep).sum()),
             novel_dropped=int((~old & ~keep).sum()))
    for c in range(off, len(names)):
        r[names[c]] = f"{int(((y == c) & ~keep).sum())}/{int((y == c).sum())}"
    rows31.append(r)
    leak.update(names[c] for c in y[old & keep])
    to_old.update((names[t], names[s]) for t, s in zip(y[~old & ~keep], slot[~old & ~keep]))
T31 = pd.DataFrame(rows31).set_index("snap")
print("old images dropped / kept (of 50); novel images dropped per class (dropped / in pool):")
print(T31.to_string())
print("\nold images kept (leaked), by class, 15 snapshots:", dict(leak.most_common()))
print("novel images dropped -> old slot, 15 snapshots:", dict(to_old.most_common(10)))

P, rows = POOL[RUN], []
for k in range(5):
    m, o = P["ids"] == k, P["y"] < off
    rows.append(dict(cluster=k, n=int(m.sum()), old=int((m & o).sum()), old_dropped=int((m & o & ~P["keep"]).sum()),
                     novel_dropped=int((m & ~o & ~P["keep"]).sum()),
                     novel_dropped_by_class=dict(Counter(names[c] for c in P["y"][m & ~o & ~P["keep"]]))))
print("\nr0 s4 per cluster:")
print(pd.DataFrame(rows).set_index("cluster").to_string())

P = POOL["gc10_v2_ft_r0_s2"]
n = np.zeros((len(names), 5), dtype=int)
np.add.at(n, (P["y"], P["ids"]), 1)
k = int(np.flatnonzero(n.argmax(0) == names.index("crease"))[0])
m = P["ids"] == k
count = lambda mask: {names[c]: int(((P["y"] == c) & mask).sum()) for c in range(len(names)) if ((P["y"] == c) & mask).any()}
print(f"\nr0 s2, cluster {k} (owner crease): before the filter {count(m)}")
print(f"r0 s2, cluster {k}, kept: {count(m & P['keep'])}")

old images dropped / kept (of 50); novel images dropped per class (dropped / in pool):
       old_dropped  old_kept  novel_dropped   silk inclusion rolled crease waist
snap                                                                            
r0_s0           39        11              8  0/420     6/135   1/17   0/32  1/94
r0_s1           44         6              8  0/420     6/135   1/17   0/32  1/94
r0_s2           45         5             10  0/420     8/135   1/17   0/32  1/94
r0_s3           42         8             11  0/420     8/135   2/17   0/32  1/94
r0_s4           48         2             11  0/420     8/135   2/17   0/32  1/94
r1_s0           39        11             16  0/420    14/135   1/17   0/32  1/94
r1_s1           44         6             13  0/420    11/135   1/17   0/32  1/94
r1_s2           44         6             16  0/420    13/135   1/17   1/32  1/94
r1_s3           41         9             11  0/420     9/135   1/17   0/32  1/94
r1_s4           48    

- **Kết quả**:
  - Ảnh cũ bị loại 39–48/50, lọt 2–11. Theo FT seed:
    - r0: 11, 6, 5, 8, 2;
    - r1: 11, 6, 6, 9, 2;
    - r2: 9, 11, 7, 5, 2.
  - Ảnh mới bị loại 5–16 mỗi ảnh chụp:
    - inclusion 4–14/135 (3.0–10.4%);
    - rolled 0–2/17; crease 0–1/32; waist 0–1/94; silk 0/420 ở 15/15.
  - Lớp cũ lọt, cộng 15 ảnh chụp: oil 39, water 29, welding 12, punching 11, crescent 9.
  - Ảnh mới bị loại rơi vào khe cũ nào, cộng 15 ảnh chụp:
    - inclusion → water 80, → oil 32, → punching 16;
    - waist → crescent 13;
    - rolled → water 9, → oil 6;
    - crease → crescent 1.
  - r0 s4 theo cụm:
    - cụm 0 loại 16/16 ảnh cũ và 1 inclusion;
    - cụm 1 (waist) có 23 ảnh cũ, loại cả 23, cộng 6 ảnh mới (inclusion 3, rolled 2, waist 1);
    - cụm 2 loại 8/10 ảnh cũ và 4 inclusion;
    - cụm 3 loại 1/1 ảnh cũ.
  - r0 s2, cụm 2 (do crease sở hữu):
    - trước bộ lọc có 34 ảnh cũ trong 93 ảnh (37%);
    - sau bộ lọc còn 2 ảnh cũ (welding 1, oil 1) trong 59 ảnh (3.4%);
    - giữ crease 28/28, rolled 13/14, waist 10/11, inclusion 4/6.
- **Ý nghĩa**:
  - Đúng ở r0 s2: bộ lọc làm sạch cụm crease (loại 32/34 ảnh cũ, giữ đủ crease). Ca ảnh cũ không còn là thiểu số trong cụm vẫn được lọc.
  - Đúng ở tỷ lệ loại của các lớp hiếm: silk 0, waist ≤ 1, crease ≤ 1, rolled ≤ 2.
  - Sai ở số ảnh cũ bị loại: dự đoán 44–50, nhưng 7/15 ảnh chụp ở mức 39–43.
  - Sai ở inclusion của r1: 13–14/135 vượt khoảng 3–8%.
  - Sai ở khe nhận ảnh: trên pool, inclusion bị khe water lấy là chính (80/128). Trên test, oil lấy là chính (`11` §4.2).
  - Sai ở r0 s4: cụm 1 có 23 ảnh cũ, không phải 19 như tôi lấy từ `09d` §2.1. Vì vậy nó mất 23 ảnh cũ và 6 ảnh mới, không phải 19 và 10.
  - Số ảnh cũ lọt đi theo seed hơn theo FT seed: s4 lọt 2 ở cả 3 FT seed, s0 lọt 9–11. Ở mỗi seed, 50 ảnh cũ được bốc giống hệt nhau giữa r0–r2 (`04-loader` §3.2). [SUY LUẬN] Ảnh cũ nào lọt phụ thuộc chính ảnh được bốc, chủ yếu ảnh oil và water.
  - [SUY LUẬN] inclusion → water khớp với nhãn đi chung: 23/135 ảnh inclusion của S1 có hộp water_spot (`00-data-gc10` §3.4; Q08.02). Một phần ảnh bị loại có thể đúng theo nội dung ảnh.

### 3.2 Bộ lọc quyết định bằng số hạng nào; bỏ log det (D25)

Tách đúng log-mật độ của từng khe trên `z_pool` với `params_f`, như `08` §3.1, bằng float64: b = −½ log det Σ, q = −½ xᵀ(Σ⁻¹ − I)x, ℓ = μᵀΣ⁻¹x, k = −½ μᵀΣ⁻¹μ.

Với mỗi ảnh, lấy khe mới tốt nhất và khe cũ tốt nhất theo tổng. Hiệu giữa hai khe là khoảng cách quyết định (margin): ảnh được giữ khi hiệu dương. Hiệu này bằng đúng Δb + Δq + Δℓ + Δk.

Báo theo ảnh chụp:
- số ảnh mà bộ lọc float64 cho cùng quyết định với `keep`;
- tỷ lệ ảnh mà |Δq| lớn nhất trong |Δb|, |Δq|, |Δℓ|;
- Δb trung bình, cùng khoảng b của khe cũ và khe mới;
- bốn bộ lọc: đủ số hạng; bỏ b (cách đọc theo paper, D25); chỉ q; chỉ ℓ. Mỗi bộ lọc báo số ảnh cũ lọt / số ảnh mới bị loại.

Dự đoán [SUY LUẬN]:
- bản float64 trùng `keep` ở ít nhất 746/748 ảnh;
- |Δq| lớn nhất ở trên 90% ảnh;
- b của khe mới lớn hơn khe cũ (`11` §2.1: det khe mới 0.14–0.38), nên Δb trung bình dương, khoảng +0.1 đến +0.5;
- bỏ b đổi 0–10 quyết định mỗi ảnh chụp, phần lớn theo hướng loại thêm;
- chỉ q gần bằng đủ số hạng;
- giả thuyết "khuôn không mang vị trí" (phần phản biện, điểm 2): bộ lọc chỉ ℓ (thành phần theo vị trí) để lọt nhiều ảnh cũ hơn hẳn bộ lọc đủ số hạng, ít nhất gấp đôi.

In [5]:
# 3.2 why the filter keeps or drops: exact split of each slot's log-density (as 08 §3.1) on the pool with params_f;
#     margin = best novel slot - best old slot = db + dq + dl + dk; filters that keep only some of the terms
def terms(p, X):
    """b (C,), q (n, C), l (n, C), k (C,) of 08 §3.1, float64."""
    mu, S = np.asarray(p["class_means"], np.float64), np.asarray(p["class_covs"], np.float64)
    Pm = np.linalg.inv(S)
    return (-0.5 * np.linalg.slogdet(S)[1], -0.5 * np.einsum("ni,cij,nj->nc", X, Pm - np.eye(X.shape[1]), X),
            np.einsum("ci,cij,nj->nc", mu, Pm, X), -0.5 * np.einsum("ci,cij,cj->c", mu, Pm, mu))

rows32, tot32 = [], Counter()
for run in RUNS:
    bl = replay.snapshots(run)
    X = np.asarray(bl["05"]["out"]["z_pool"], np.float64)
    b, q, l, k = terms(bl["11"]["out"]["params_f"], X)
    B, K = np.broadcast_to(b, q.shape), np.broadcast_to(k, q.shape)
    full = B + q + l + K
    old, keep = POOL[run]["y"] < off, POOL[run]["keep"]
    jn, jo = off + full[:, off:].argmax(1), full[:, :off].argmax(1)          # best novel slot, best old slot
    ii = np.arange(len(X))
    d = np.stack([T[ii, jn] - T[ii, jo] for T in (B, q, l, K)], 1)           # db, dq, dl, dk; sum = margin
    r = dict(snap=run.replace("gc10_v2_ft_", ""),
             same_as_keep=int(((full[:, off:].max(1) > full[:, :off].max(1)) == keep).sum()),
             q_largest=round(float((np.abs(d[:, :3]).argmax(1) == 1).mean()), 3), mean_db=round(float(d[:, 0].mean()), 3),
             b_old=f"{b[:off].min():.2f}-{b[:off].max():.2f}", b_new=f"{b[off:].min():.2f}-{b[off:].max():.2f}")
    for name, T in (("full", full), ("no_b", q + l + K), ("q", q), ("l", l)):
        kp = T[:, off:].max(1) > T[:, :off].max(1)
        ok_, nd = int((old & kp).sum()), int((~old & ~kp).sum())
        r[name] = f"{ok_}/{nd}"
        tot32[(name, "old kept")] += ok_
        tot32[(name, "novel dropped")] += nd
    rows32.append(r)
T32 = pd.DataFrame(rows32).set_index("snap")
print("same_as_keep: float64 filter = snapshot keep (of 748); q_largest: share of images where |dq| is the largest of |db|, |dq|, |dl|")
print("mean_db: b(best novel slot) - b(best old slot); columns full / no_b (paper, D25) / q / l: old kept / novel dropped")
print(T32.to_string())
print("\ntotals over 15 snapshots (old kept of 750, novel dropped of 10470):",
      {f"{n}: {w}": v for (n, w), v in tot32.items()})

same_as_keep: float64 filter = snapshot keep (of 748); q_largest: share of images where |dq| is the largest of |db|, |dq|, |dl|
mean_db: b(best novel slot) - b(best old slot); columns full / no_b (paper, D25) / q / l: old kept / novel dropped
       same_as_keep  q_largest  mean_db      b_old      b_new   full   no_b      q      l
snap                                                                                     
r0_s0           748      0.991    0.355  0.41-0.71  0.31-0.95   11/8   12/9   13/9  10/22
r0_s1           748      0.993    0.360  0.41-0.72  0.30-0.95    6/8   6/10    6/9   9/21
r0_s2           748      0.989    0.354  0.41-0.70  0.31-0.92   5/10   5/11   7/12   7/26
r0_s3           748      0.997    0.364  0.41-0.71  0.49-0.96   8/11   8/12   7/12   9/28
r0_s4           748      0.996    0.359  0.41-0.71  0.49-0.98   2/11   2/12   2/12   3/27
r1_s0           748      0.985    0.358  0.32-0.63  0.34-0.96  11/16  11/18  11/17   7/36
r1_s1           748      0.985    0.3

- **Kết quả**:
  - Bộ lọc tính lại bằng float64 trùng `keep` ở 748/748 ảnh, cả 15 ảnh chụp.
  - |Δq| lớn nhất trong ba hiệu ở 98.4–100% ảnh.
  - b của khe cũ là 0.31–0.72, của khe mới là 0.30–0.98. Δb trung bình (khe mới tốt nhất − khe cũ tốt nhất) là +0.34 đến +0.37.
  - Cộng 15 ảnh chụp, số ảnh cũ lọt / số ảnh mới bị loại của bốn bộ lọc:

    | bộ lọc | ảnh cũ lọt (/750) | ảnh mới bị loại (/10470) |
    |---|---|---|
    | đủ số hạng (pipeline) | 100 | 157 |
    | bỏ b (paper, D25) | 96 | 179 |
    | chỉ q | 102 | 177 |
    | chỉ ℓ | 138 | 408 |

  - Từng ảnh chụp, bỏ b đổi số ảnh cũ lọt −3 đến +1 và số ảnh mới bị loại 0 đến +4.
- **Ý nghĩa**:
  - Đúng ở phép kiểm (748/748), ở q quyết định (trên 98%), ở dấu và độ lớn của Δb, và ở hiệu ứng nhỏ của việc bỏ b, theo hướng loại thêm.
  - D25 ở bước 12: log det cộng cho khe mới khoảng +0.35. Bỏ nó đi thì cộng 15 ảnh chụp có thêm 22 ảnh mới bị loại và bớt 4 ảnh cũ lọt. Hiệu ứng nhỏ, như ở S0 (`08` §3.2), vì q trải rộng hơn hẳn.
  - Sai ở giả thuyết "khuôn không mang vị trí". Dự đoán là bộ lọc chỉ ℓ để lọt ít nhất gấp đôi; thực tế chỉ ×1.38 (138 so với 100), và ở r0 s4 là 3 so với 2. Như vậy ℓ, tức hướng của μ, một mình cũng tách ảnh cũ khá tốt. Điều nó làm hỏng là ảnh mới: bộ lọc chỉ ℓ loại 408 ảnh mới, gấp 2.6 lần.
  - [SUY LUẬN] Bộ lọc chạy được nhờ q, nhưng không phải vì ℓ "mù vị trí". Vai trò của b và ℓ trong bộ lọc là giữ ảnh mới: chỉ q loại 177 ảnh mới so với 157 khi đủ số hạng. Lời giải thích ở phần phản biện (điểm 2) cần sửa lại theo hướng này.

## 4. Cột 12 của sổ mất điểm

### 4.1 Cột 12 trên pool, và độ sạch nhãn của tập học

Gọi `score.ledger` (cột 09, 10, 11, 12; commit `4f8166d`) cho 15 ảnh chụp. Cột 12 trên tập `pool` theo dòng Sửa của QD24:
- ảnh cũ đúng khi bị loại;
- ảnh mới đúng khi được giữ và nằm trong cụm mang tên lớp của nó.

In `ledger_summary` của pool ở r0 s4 cho các cột 09, 10, 12. Kiểm: số ảnh cũ đúng ở cột 12 phải bằng số ảnh cũ bị loại ở 3.1 (15/15).

Cột 12 chỉ có luồng mất, nên đo thêm hai thứ:
- trong các ảnh mới bị loại, bao nhiêu ảnh nằm ở cụm đúng tên (luồng mất 10 → 12) và bao nhiêu ở cụm sai tên;
- độ sạch nhãn: tỷ lệ ảnh mà cụm mang đúng tên lớp của ảnh (ảnh cũ tính là sai), tính trên 748 ảnh mà lượt F học (`purity_F`) và trên các ảnh mà lượt S học (`purity_S`). In thêm theo cụm ở r0 s4.

Dự đoán [SUY LUẬN]:
- kiểm đạt 15/15;
- r0 s4: luồng mất 10 → 12 là 2 ảnh (inclusion 1 ở cụm 0, waist 1 ở cụm 1); 9 ảnh mới bị loại còn lại nằm ở cụm sai tên (suy từ bảng theo cụm của 3.1 và tên cụm của `10` §1.1);
- trên 15 ảnh chụp, ít nhất 2/3 số ảnh mới bị loại nằm ở cụm sai tên, vì inclusion phần lớn nằm trong cụm silk;
- `purity_S` − `purity_F` là +3 đến +6 điểm, chủ yếu nhờ bỏ ảnh cũ; balanced của pool đổi dưới 1 điểm từ 10 sang 12.

In [6]:
# 4.1 loss ledger with column 12 (pool): an old image is right when dropped, a novel image when kept and in the cluster
#     named after its class; label purity of what pass F learns (748 images) and of what pass S learns (kept images)
import time
import score
t0 = time.time()
LEDGER = {run: score.ledger(run) for run in RUNS}
print(f"ledger of 15 snapshots: {time.time() - t0:.0f} s")
print("\nr0 s4, pool, novel classes:")
print(score.ledger_summary(LEDGER[RUN]["pool"], names, ["09", "10", "12"]).round(3).to_string())

rows41 = []
for run in RUNS:
    p, keep = LEDGER[run]["pool"], POOL[run]["keep"]
    right = (p["novel"] & (p["10"] == 1)).to_numpy()                 # the image's cluster carries its class name
    s_all = score.ledger_summary(p, names, ["10", "12"])
    s = s_all.drop(index="balanced")
    tag = run.replace("gc10_v2_ft_", "")
    rows41.append(dict(snap=tag, old_right_12=int(p.loc[~p["novel"], "12"].sum()),
                       check=bool(int(p.loc[~p["novel"], "12"].sum()) == int(T31.loc[tag, "old_dropped"])),
                       novel_dropped=int((p["novel"].to_numpy() & ~keep).sum()), dropped_right_named=int(s["lost_10_12"].sum()),
                       purity_F=round(float(right.mean()), 3), purity_S=round(float(right[keep].mean()), 3),
                       bal_10=round(float(s_all.loc["balanced", "10"]), 3), bal_12=round(float(s_all.loc["balanced", "12"]), 3)))
T41 = pd.DataFrame(rows41).set_index("snap")
T41["gain"] = (T41["purity_S"] - T41["purity_F"]).round(3)
print("\n15 snapshots, pool:")
print(T41.to_string())
print("\nmean by FT seed:")
print(T41[["purity_F", "purity_S", "gain", "bal_10", "bal_12"]].groupby(T41.index.str[:2]).mean().round(3).to_string())

p, keep = LEDGER[RUN]["pool"], POOL[RUN]["keep"]
right, cl = (p["novel"] & (p["10"] == 1)).to_numpy(), p["cluster"].to_numpy()
print("\nr0 s4, purity per cluster (pass F -> pass S):",
      {k: (round(float(right[cl == k].mean()), 3), round(float(right[(cl == k) & keep].mean()), 3)) for k in range(5)})

ledger of 15 snapshots: 3 s

r0 s4, pool, novel classes:
               n       09       10  lost_09_10  gained_09_10       12  lost_10_12  gained_10_12
silk       420.0  390.000  169.000       221.0           0.0  169.000         0.0           0.0
inclusion  135.0   46.000   46.000         0.0           0.0   45.000         1.0           0.0
rolled      17.0    0.000    0.000         0.0           0.0    0.000         0.0           0.0
crease      32.0    0.000    0.000         0.0           0.0    0.000         0.0           0.0
waist       94.0   79.000   79.000         0.0           0.0   78.000         1.0           0.0
balanced     NaN    0.422    0.317         NaN           NaN    0.313         NaN           NaN

15 snapshots, pool:
       old_right_12  check  novel_dropped  dropped_right_named  purity_F  purity_S  bal_10  bal_12   gain
snap                                                                                                     
r0_s0            39   True            

- **Kết quả**:
  - Sổ mất điểm của 15 ảnh chụp tính trong 3 s. Ở cột 12, số ảnh cũ đúng (tức bị loại) trùng số ảnh cũ bị loại của 3.1 ở 15/15.
  - r0 s4, pool, từ cột 10 sang 12:
    - inclusion 46 → 45 và waist 79 → 78 (mỗi lớp mất 1 ảnh);
    - silk 169, rolled 0, crease 0 không đổi;
    - balanced 0.317 → 0.313.
  - Ảnh mới bị loại nằm ở cụm mang đúng tên lớp của chúng: 1–5 ảnh mỗi ảnh chụp, cộng 32/157 (20%). 80% còn lại nằm ở cụm sai tên.
  - Độ sạch nhãn:
    - `purity_F` 0.274–0.453, `purity_S` 0.291–0.483, tăng +1.7 đến +3.1 điểm;
    - trung bình theo FT seed (r0 / r1 / r2): 0.391 → 0.418, 0.363 → 0.390, 0.376 → 0.400.
  - Balanced của pool giảm 0.1–0.8 điểm từ cột 10 sang 12.
  - r0 s4 theo cụm (lượt F → lượt S):
    - cụm 0: 0.484 → 0.577; cụm 1: 0.503 → 0.609; cụm 2: 0.621 → 0.650;
    - cụm 3 và cụm 4 (silk, mang tên rolled và crease): 0 → 0.
- **Ý nghĩa**:
  - Đúng ở phép kiểm, ở r0 s4 (luồng mất là 2 ảnh, inclusion 1 và waist 1), ở tỷ lệ ảnh bị loại nằm ở cụm sai tên (80% ≥ 2/3), và ở mức balanced đổi dưới 1 điểm.
  - Sai ở độ lớn của độ sạch: tăng +1.7 đến +3.1 điểm, dưới khoảng dự đoán +3 đến +6.
  - Trước bộ lọc, chỉ 27–45% nhãn mà lượt F học là đúng tên. Bộ lọc nâng con số này thêm 2–3 điểm. Phần lớn ảnh mà nó loại nằm ở cụm sai tên, nên nó ít khi làm mất ảnh tốt.
  - [SUY LUẬN] Nhiễu nhãn của lượt S đến chủ yếu từ bước 09–10, không từ ảnh cũ. Ở r0 s4, hai cụm silk mang tên rolled và crease (224 ảnh) có độ sạch 0, còn ảnh cũ chỉ là 50/748. Bộ lọc không phải chỗ mất điểm; cải tiến bộ lọc có trần thấp.

## Summary (EN)

1. **Step 12 replays exactly, and it only masks.** `rerun` gives `keep` identical to the snapshot (689/748 kept at r0 s4). Kept counts equal `S1_novel_kept` in 15/15 snapshots (§1.1). Pass S keeps the cluster ids of the kept images (D12). On the pool, the pass-F classifier agrees with the k-means ids on 89.6–92.6% of the images and puts only 7–21 kept images into another novel slot. The paper's re-labeling (Algorithm 1, lines 34–39) would therefore change few novel labels; the real difference from the paper is the 46–61 dropped images (§2.1).
2. **The filter removes most old images and few novel ones.** It drops 39–48 of the 50 old images and 5–16 of the 698 novel ones: inclusion 4–14/135, rolled ≤ 2, crease ≤ 1, waist ≤ 1, silk 0. Leaked old images are mostly oil (39) and water (29) over 15 snapshots, and the leak count follows the seed (which old images are drawn), not the FT seed. Dropped novel images go mostly inclusion → water_spot (80 of 157). At r0 s2, the crease-owned cluster falls from 34/93 old images to 2/59 and keeps all 28 crease images (§3.1).
3. **The quadratic term decides; log det barely matters (D25).** |Δq| is the largest term of the keep/drop margin on 98.4–100% of the pool images. The log det term favours the novel slots by +0.35 on average, yet dropping it, as the paper's Eq. 4/6 read, changes only 22 novel drops and 4 leaks over 15 snapshots. A filter on the linear term alone leaks 1.38× as many old images but drops 2.6× as many novel ones (§3.2). [inference] b and ℓ mainly keep novel images; the "location-blind template" explanation of why the SVI filter works is not supported.
4. **Ledger column 12: the filter is not where points are lost.** Of the dropped novel images, 80% sit in wrongly named clusters. Label purity of the training set rises only from 0.27–0.45 (pass F) to 0.29–0.48 (pass S), +1.7 to +3.1 points. Pool balanced changes by under 1 point (§4.1). [inference] Most of the label noise that pass S learns comes from steps 09–10: at r0 s4, the two silk clusters named rolled and crease have purity 0.